# Pipeline Step 06 — Eksi Sozluk LLM Annotation

Annotates Eksi Sozluk entries that were not covered by human annotators
using the Claude API (few-shot classification).

## What this notebook does
- Loads the human-annotated entries (annotator A and annotator B)
- Builds few-shot examples from agreed human annotations
- Calls `claude-haiku-4-5-20251001` to label all remaining entries
- Saves results incrementally (checkpoint every 100 entries)

## Inputs
- `data/raw/eksi/eksi_corpus_final.csv` — full corpus from step 05
- `data/processed/eksi_annotation_v2_labeled_a.csv` — human annotations (annotator A)
- `data/processed/eksi_annotation_v2_labeled_b.csv` — human annotations (annotator B)

## Output
- `data/processed/eksi_llm_annotated.csv` — LLM labels for remaining entries

## Setup
1. `pip install anthropic pandas tqdm`
2. Set `ANTHROPIC_API_KEY` as an environment variable
3. Run `TEST_MODE = True` first to verify labels look correct
4. Set `TEST_MODE = False` for the full run (~3 hours for ~4,000 entries)

## Annotation schema
Each entry receives 5 binary labels:
- `physical_violence` — specific physical attack described
- `verbal_violence` — specific verbal attack described
- `violence_normalized` — violence explicitly justified or normalised
- `anti_violence` — explicit opposition to violence
- `sarcasm` — genuine sarcasm/irony present

In [ ]:
# !pip install anthropic pandas tqdm -q

In [ ]:
from getpass import getpass
import os

# Set API key from environment or prompt interactively
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")

In [ ]:
import json
import os
import time
from pathlib import Path

import anthropic
import pandas as pd
from tqdm import tqdm

# ── PATH CONFIG ───────────────────────────────────────────────────────────────
INPUT_CORPUS  = Path("data/raw/eksi/eksi_corpus_final.csv")
INPUT_ANNOT_A = Path("data/processed/eksi_annotation_v2_labeled_a.csv")
INPUT_ANNOT_B = Path("data/processed/eksi_annotation_v2_labeled_b.csv")
OUTPUT_PATH   = Path("data/processed/eksi_llm_annotated.csv")
# ─────────────────────────────────────────────────────────────────────────────

MODEL      = "claude-haiku-4-5-20251001"
TEST_MODE  = True   # set False for full run after test passes

MAX_RETRIES      = 3
RETRY_DELAY      = 5    # seconds between retry attempts
RATE_LIMIT_SLEEP = 0.5  # seconds between requests

COLS = [
    "physical_violence",
    "verbal_violence",
    "violence_normalized",
    "anti_violence",
    "sarcasm",
]

In [ ]:
# Annotation guideline sent as part of the LLM prompt.
# Must remain in Turkish — the model scores Turkish-language entries.
GUIDELINE = """
Sen bir sosyal bilimler arastirmasi icin Turkce Eksi Sozluk entry'lerini etiketliyorsun.
Arastirma konusu: Turkiye'de saglik calisanlarina yonelik siddetin Eksi Sozluk'te nasil
tartisildigini olcmek.

Her entry icin 5 binary (0/1) etiket belirle:

1. physical_violence: Hasta veya hasta yakini tarafindan saglik calisanina yonelik somut
   fiziksel saldiri anlatiliyor veya aktariliyor mu?
   1 = spesifik fiziksel saldiri olayi var | 0 = yok veya sadece genel/sistemik bahis

2. verbal_violence: Hasta veya hasta yakini tarafindan saglik calisanina yonelik sozel
   saldiri (tehdit, hakaret, bagirma) anlatiliyor mu?
   1 = spesifik sozel saldiri var | 0 = yok

3. violence_normalized: Entry siddeti ACIKCA mesrulastiriyor veya hakli gosteriyor mu?
   1 = sadece "hak etti", "yapilabilir", "anlayisla karsiliyorum" gibi acik mesrulastirma
   0 = sistemik elestiri, doktoru suclama, dolayli ima -- bunlar 0 alir

4. anti_violence: Entry siddeti acikca kiniyor veya karsi cikiyor mu?
   1 = siddete karsi oldugunu acikca ifade ediyor | 0 = yok veya belirsiz

5. sarcasm: Entry ironi veya sarcasm iceriyor mu?
   1 = sadece gercek anlami soylenenlerden acikca zitsa ("kesinlikle ise yarar" gibi)
   0 = belirsiz durumlarda 0 ver

KURALLAR:
- Sadece entry metnini degerlendir, thread adina bakma
- Mobbing veya meslektastan gelen siddet 1 almaz
- Sistemik sikayet (maas, calisma kosullari) violence_normalized almaz
- Sarcasm varsa once isaretcle, diger kolonlari gercek anlama gore doldur
- Her kolon bagimsiz, birden fazla 1 olabilir
"""

In [ ]:
def build_few_shot(annot_a: pd.DataFrame, annot_b: pd.DataFrame) -> str:
    """
    Builds few-shot examples from agreed human annotations.
    Uses annotator B's labels as ground truth for disagreements
    (annotator B applied the more conservative labelling standard).
    Returns a formatted string with ~7 examples covering all label categories.
    """
    merged = annot_a[["entry_id", "content"] + COLS].merge(
        annot_b[["entry_id"] + COLS],
        on="entry_id",
        suffixes=("_a", "_b"),
    ).dropna()

    for col in COLS:
        merged[f"{col}_a"] = merged[f"{col}_a"].astype(int)
        merged[f"{col}_b"] = merged[f"{col}_b"].astype(int)

    agree_mask = True
    for col in COLS:
        agree_mask = agree_mask & (merged[f"{col}_a"] == merged[f"{col}_b"])
    agreed = merged[agree_mask].copy()

    # Use annotator B's labels as ground truth
    for col in COLS:
        agreed[col] = agreed[f"{col}_b"]

    examples = []

    # All-zero examples
    examples.append(agreed[(agreed[COLS] == 0).all(axis=1)].head(1))

    # Physical violence (agreed)
    examples.append(agreed[agreed["physical_violence"] == 1].head(1))

    # Anti-violence only (agreed, no physical)
    examples.append(
        agreed[
            (agreed["anti_violence"] == 1) & (agreed["physical_violence"] == 0)
        ].head(1)
    )

    # Violence normalised — use annotator B positives (clearest cases)
    b_norm = merged[merged["violence_normalized_b"] == 1].copy()
    for col in COLS:
        b_norm[col] = b_norm[f"{col}_b"]
    examples.append(b_norm.head(2))

    # Sarcasm — use annotator B positives
    b_sarc = merged[merged["sarcasm_b"] == 1].copy()
    for col in COLS:
        b_sarc[col] = b_sarc[f"{col}_b"]
    examples.append(b_sarc.head(2))

    combined = (
        pd.concat(examples)[["entry_id", "content"] + COLS]
        .drop_duplicates(subset="entry_id")
        .head(7)
    )

    shots = []
    for _, row in combined.iterrows():
        label_dict = {col: int(row[col]) for col in COLS}
        shots.append(
            f"Entry: {str(row['content'])[:400]}\n"
            f"Etiket: {json.dumps(label_dict, ensure_ascii=False)}"
        )

    return "\n\n".join(shots)

In [ ]:
def annotate_entry(
    client: anthropic.Anthropic,
    content: str,
    few_shot_text: str,
) -> dict | None:
    """
    Sends a single entry to the Claude API and parses the JSON label response.
    Returns a dict of {col: 0|1} or None on repeated failure.
    """
    prompt = (
        f"{GUIDELINE}\n\n"
        f"Asagida ornek entry'ler ve dogru etiketleri var:\n\n"
        f"{few_shot_text}\n\n"
        f"Simdi su entry'yi etiketle. SADECE JSON dondur, baska hicbir sey yazma.\n"
        f'Format: {{"physical_violence": 0, "verbal_violence": 0, '
        f'"violence_normalized": 0, "anti_violence": 0, "sarcasm": 0}}\n\n'
        f"Entry: {content[:600]}\nEtiket:"
    )

    for attempt in range(MAX_RETRIES):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=80,
                messages=[{"role": "user", "content": prompt}],
            )
            raw = response.content[0].text.strip()

            if "{" in raw:
                raw = raw[raw.index("{") : raw.rindex("}") + 1]

            labels = json.loads(raw)

            for col in COLS:
                if col not in labels:
                    raise ValueError(f"Missing key in response: {col}")
                labels[col] = int(bool(labels[col]))

            return labels

        except Exception as e:
            print(f"  Attempt {attempt + 1} failed: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_DELAY)

    return None

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
corpus   = pd.read_csv(INPUT_CORPUS,  encoding="utf-8-sig")
annot_a  = pd.read_csv(INPUT_ANNOT_A, sep=None, engine="python")
annot_b  = pd.read_csv(INPUT_ANNOT_B, sep=None, engine="python")

print(f"Corpus       : {len(corpus)} entries")
print(f"Annotator A  : {len(annot_a)} entries")
print(f"Annotator B  : {len(annot_b)} entries")

# IDs already annotated by humans
annotated_ids = set(annot_a["entry_id"]).union(set(annot_b["entry_id"]))

# Resume if partial output exists
existing = pd.DataFrame()
if OUTPUT_PATH.exists():
    existing = pd.read_csv(OUTPUT_PATH)
    annotated_ids.update(existing["entry_id"].tolist())
    print(f"Resuming     : {len(annotated_ids)} total annotated so far")

remaining = corpus[~corpus["entry_id"].isin(annotated_ids)].copy()
print(f"Remaining    : {len(remaining)} entries")

if TEST_MODE:
    remaining = remaining.head(10)
    print(f"TEST MODE    : running on {len(remaining)} entries only")

# ── Build few-shot examples ───────────────────────────────────────────────────
few_shot_text = build_few_shot(annot_a, annot_b)
n_shots = few_shot_text.count("Entry:")
print(f"Few-shot examples: {n_shots}")

# ── Annotate ─────────────────────────────────────────────────────────────────
client  = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
results = []
failed  = []

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

for _, row in tqdm(remaining.iterrows(), total=len(remaining), desc="Annotating"):
    labels = annotate_entry(client, str(row["content"]), few_shot_text)

    if labels is not None:
        record = {"entry_id": row["entry_id"]}
        record.update(labels)
        results.append(record)

        if TEST_MODE:
            print(f"\n  entry_id : {row['entry_id']}")
            print(f"  content  : {str(row['content'])[:150]}")
            print(f"  labels   : {labels}")
    else:
        failed.append(row["entry_id"])
        print(f"  FAILED: entry_id {row['entry_id']}")

    # Save checkpoint every 100 entries
    if not TEST_MODE and len(results) % 100 == 0 and len(results) > 0:
        ckpt = pd.DataFrame(results)
        if not existing.empty:
            ckpt = pd.concat([existing, ckpt], ignore_index=True)
        ckpt.to_csv(OUTPUT_PATH, index=False)
        print(f"  Checkpoint: {len(results)} entries saved")

    time.sleep(RATE_LIMIT_SLEEP)

# ── Final save ────────────────────────────────────────────────────────────────
final = pd.DataFrame(results)
if not existing.empty:
    final = pd.concat([existing, final], ignore_index=True)
final.to_csv(OUTPUT_PATH, index=False)

print(f"\nDone — annotated: {len(results)}, failed: {len(failed)}")
if failed:
    print(f"Failed IDs: {failed}")
if TEST_MODE:
    print("\nTest passed. Set TEST_MODE = False for full run.")